# Frequenzmodelle: GLM, GAM und Histogramm-Poisson

Dieses Notebook liest und prüft die gespeicherten Ergebnisse von `python -m src.frequency`. Es passt kein Modell an. Alle Vorhersagen und Auswahlwerte beziehen sich auf die Entwicklungsverträge des eingefrorenen Splits. Die gemeinsame Evaluation mit der späteren Schadenhöhenanalyse bleibt geschlossen. Die frühere Exploration des ganzen Datenstands bleibt offengelegt; dies ist keine externe Validierung.

[Modellspezifikationen und Grenzen](../docs/frequenzmodelle.md), [registriertes Vergleichsprotokoll](../config/frequency_protocol.json). Anzahlen und Exponierungen bleiben ungekappt; die Zielgröße ist die Anzahl erfasster positiver Schadenzeilen je Vertragsperiode.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image

start = Path.cwd().resolve()
ROOT = next(p for p in [start, *start.parents] if (p / 'config/frequency_protocol.json').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.frequency import load_report
from src.data import sha256

report, tables = load_report()
local = ROOT / 'data/frequency' / report['run_id']
assert sha256(local / 'registration.json') == report['registration_sha256']
for model_id, model_report in report['models'].items():
    model_dir = local / model_id
    manifest = json.loads((model_dir / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest == model_report
    assert sha256(model_dir / 'oof_predictions.parquet') == model_report['artifacts']['oof_predictions']['sha256']
for model_id, refit in report['development_refits'].items():
    assert json.loads((local / f'{model_id}.refit.json').read_text(encoding='utf-8')) == refit
    assert sha256(local / f'{model_id}.joblib') == refit['sha256']
source = json.loads((ROOT / 'reports/s1/preparation_report.json').read_text(encoding='utf-8'))
split = json.loads((ROOT / 'reports/s2/split_manifest.json').read_text(encoding='utf-8'))
assert report['evaluation_predictions'] == 0 and report['evaluation_scores'] == 0
assert report['source_id'] == source['source_id']
assert report['preparation_id'] == source['preparation_id']
assert report['split_id'] == split['split_id']
decision = report['decision']
assert decision['evaluation_released'] is False
champions = list(decision['family_winners'].values())
selected = decision['selected_frequency_model']
print('Lauf:', report['run_id'])
print('Quelle:', report['source_id'], '| Aufteilung:', report['split_id'])
print('Lokale Tabellen, öffentliche Spiegel, Bild- und Code-Hashes geprüft.')
print('Entwicklungsentscheidung:', selected, '| Familienkandidaten:', ', '.join(champions))

## Vergleich und Entwicklungsentscheidung

Das Exponierungs-Basismodell, drei GLM-Spezifikationen, vier GAM-Spezifikationen und zwei Histogramm-Poisson-Konfigurationen verwenden dieselben fünf Entwicklungsfalten. Jede Vertragsvorhersage stammt aus einem Fit ohne die eigene Falte. Kodierung, Skalierung und Splinebasen werden je Trainingsfalte gelernt.

Der primäre Score ist `sum d(N, Exposure × Jahresrate) / sum Exposure`. Erwartete Periodenanzahlen werden nicht nochmals mit Exponierung gewichtet. A/P bedeutet beobachtete durch vorhergesagte Anzahl; Gini bezeichnet rohe Frequenzkonzentration nach absteigender Jahresrate. Die durchschnittliche quadratische OOF-Pearson-Abweichung ist eine Vorhersagediagnostik mit Nenner Vertragszahl, keine anhand von Residualfreiheitsgraden geschätzte Dispersion.

In [ ]:
GERMAN = {
    'model_id': 'Modell', 'family': 'Familie', 'policies': 'Verträge',
    'exposure_years': 'Vertragsjahre', 'observed_claims': 'Beobachtete Schäden',
    'expected_claims': 'Erwartete Periodenanzahl',
    'observed_annual_rate': 'Beobachtete Schäden/Jahr',
    'predicted_annual_rate': 'Vorhergesagte Schäden/Jahr',
    'actual_over_predicted': 'A/P', 'poisson_deviance_per_exposure': 'Poisson-Deviance/Vertragsjahr',
    'mean_count_deviance': 'Mittlere Anzahl-Deviance', 'raw_frequency_gini': 'Roher Frequenz-Gini',
    'pearson_mean_quadratic': 'Mittlere quadratische OOF-Pearson-Abweichung',
    'relative_deviance_reduction_vs_intercept': 'Relative Deviancereduktion zum Basismodell',
    'fold': 'Falte', 'reference_model': 'Referenzmodell', 'deviance_difference': 'Deviancedifferenz',
    'lower': 'Perzentil unten', 'upper': 'Perzentil oben', 'replicates': 'Ziehungen',
    'dimension': 'Merkmal', 'level': 'Ausprägung', 'supported': 'Mindestens 500 Vertragsjahre',
    'assigned_age': 'Gesetztes Fahreralter', 'standardized_annual_rate': 'Standardisierte Jahresrate',
    'relativity_to_first_age': 'Relativität zum ersten Alter',
    'reference_policies': 'Referenzverträge', 'reference_exposure_years': 'Referenzvertragsjahre',
    'feature': 'Merkmal', 'value': 'Wert', 'relativity': 'Relativität',
    'exposure_at_exact_value': 'Vertragsjahre beim exakten Wert'
}
def show(frame):
    display(frame.rename(columns=GERMAN))

show(tables['scores'].sort_values('poisson_deviance_per_exposure'))
display(Image(filename=str(ROOT / 'reports/s4/figures/modellvergleich.png')))
print('Auswahlregel:', decision['rule'])
print('Begründung:', decision['rationale'])

## Falten und bedingte Stichprobenstabilität

Die Faltenscores haben unterschiedliche Exponierungsnenner. Ihr korrekt gepoolter Score ist das mit Exponierung gewichtete Mittel. Fünf überlappende Trainingsfits sind keine unabhängigen Wiederholungen zur Schätzung eines Standardfehlers.

Die 500 gepaarten Bootstrap-Ziehungen verwenden ganze Verträge und dieselben gezogenen Indizes für alle festen OOF-Vorhersagen. Jede Ziehung wird durch ihre gezogene Exponierungssumme geteilt. Die Differenz lautet Kandidat minus Referenz; negative Werte begünstigen den Kandidaten. Die Perzentilintervalle beschreiben bedingte Stichprobenstabilität. Sie enthalten weder erneutes Fitten noch Modellwahl oder unbekannte Haushaltsabhängigkeit. Zur Auswahl benutzte OOF-Werte bewerten nicht unverzerrt das gesamte abgestimmte Verfahren.

In [ ]:
focus = list(dict.fromkeys(['intercept', *champions, selected]))
show(tables['folds'].loc[tables['folds'].model_id.isin(focus),
     ['model_id', 'fold', 'policies', 'exposure_years', 'observed_claims',
      'expected_claims', 'actual_over_predicted', 'poisson_deviance_per_exposure']])
show(tables['paired_bootstrap'].loc[tables['paired_bootstrap'].model_id.isin(focus),
     ['model_id', 'reference_model', 'deviance_difference', 'lower', 'upper', 'replicates', 'seed']])

## Kalibrierung nach Merkmalen und Exponierung

Die Tabellen zeigen Alters-, Bonus-Malus-, Kraftstoff-, Regions-, Exponierungs- und B12-Gruppen. Jede Zeile enthält beobachtete und erwartete Anzahl, Vertragsjahre sowie A/P. Die Grenze von 500 Vertragsjahren kennzeichnet geringe Unterstützung; sie begründet weder eine Signifikanzschwelle noch die Richtigkeit eines Modells.

Die Vertragsdauer ist kein Modellmerkmal. Abweichungen nach Exponierungsgruppen prüfen die Annahme einer zu Exponierung proportionalen bedingten Anzahl unter den gewählten Merkmalen. Sie identifizieren keinen Beendigungsmechanismus. B12 bleibt erhalten; die anonymisierte Klasse ist kein bestätigter Datenfehler.

In [ ]:
segment_columns = ['model_id', 'dimension', 'level', 'policies', 'exposure_years',
                   'observed_claims', 'expected_claims', 'observed_annual_rate',
                   'predicted_annual_rate', 'actual_over_predicted', 'supported']
for dimension in ['age_band', 'bm_band', 'VehGas', 'Region', 'exposure_band', 'brand12', 'brand_new']:
    print('Merkmal:', dimension)
    show(tables['segments'].loc[
        tables['segments'].model_id.isin(champions) & tables['segments'].dimension.eq(dimension),
        segment_columns])
show(tables['calibration'].loc[tables['calibration'].model_id.isin(focus),
     ['model_id', 'group', 'policies', 'exposure_years', 'observed_claims',
      'expected_claims', 'observed_annual_rate', 'predicted_annual_rate', 'actual_over_predicted']])

## Gemeinsame Referenzverteilung und Modellabhängigkeit

Die Modelle werden nach der Entwicklungswahl auf allen Entwicklungsverträgen neu angepasst. Bei der Altersstandardisierung bleiben dieselben übrigen Merkmale und Exponierungsgewichte erhalten; nur das aufgezeichnete Alter wird auf 35, 45 oder 55 gesetzt. Die Referenz enthält Verträge im Alter 30–59 aus Bonus-Malus × Fahrzeugalter × Kraftstoff × Region-Zellen mit mindestens 100 Vertragsjahren in **jeder** der Altersgruppen 30–39, 40–49 und 50–59.

Ausgeschlossene Unterstützung wird vollständig bilanziert. Diese gröbere gemeinsame Unterstützung garantiert keine Überlappung sämtlicher kontinuierlicher Merkmale oder genauer Marken-/Dichtekombinationen. Die Punktvorhersagen sind eine deskriptive Modellstandardisierung ohne Konfidenzbänder oder kausale Identifikation. Sie sind weder marginale beobachtete Altersraten noch ein direkt mit den anders zusammengesetzten S3-BM-Raten vergleichbarer Effekt.

In [ ]:
display(pd.Series(report['age_reference_support'], name='Gemeinsame Altersreferenz'))
show(tables['standardized_age'])
cells = tables['standardization_cells']
display(cells.groupby(['age_support_band', 'common_support'], observed=True).agg(
    Zellen=('exposure_years', 'size'), Verträge=('policies', 'sum'),
    Vertragsjahre=('exposure_years', 'sum'), Schäden=('observed_claims', 'sum')).reset_index())
dependence = report['dependence']
display(pd.DataFrame(dependence['exposure_weighted_pearson_correlation'],
                     index=dependence['variables'], columns=dependence['variables']))
print('Anteil der logDensity-Variation zwischen Area-Gruppen:',
      dependence['logDensity_between_Area_variance_fraction'])
print('Anteil der logDensity-Variation zwischen Region-Gruppen:',
      dependence['logDensity_between_Region_variance_fraction'])
display(Image(filename=str(ROOT / 'reports/s4/figures/kalibrierung_standardisierung.png')))

## Relativitäten und Konzentration richtig lesen

Die Effektprofile ändern jeweils ein Merkmal an einer ausdrücklich gespeicherten Referenzzeile. Bei einem additiven Modell mit Log-Link ist das Verhältnis zweier Jahresraten `exp(Δη)`. Eine Interaktion macht dieses Verhältnis von den übrigen Merkmalen abhängig. Das Verhältnis standardisierter Mittel ist bei allgemeinen Interaktionen kein einzelner Koeffizient.

Alter, Bonus-Malus, Gebietsklasse und Dichte teilen Variation. Einzelfunktionen sind deshalb von Modell, Referenz und Unterstützung abhängig. Die Tabellen enthalten exakte Randunterstützung; sie beweist keine gemeinsame Unterstützung der gesamten Referenzkombination. Die beiden Altersachsen sind korrekt benannt. Konzentration misst Rangordnung und ersetzt keine Kalibrierung.

In [ ]:
effects = tables['effects']
chosen_points = (
    (effects.feature.eq('DrivAge') & effects.value.isin(['35', '45', '55'])) |
    (effects.feature.eq('VehAge') & effects.value.isin(['0', '1', '5', '10', '20'])) |
    (effects.feature.eq('BonusMalus') & effects.value.isin(['50', '75', '100', '150'])) |
    effects.feature.isin(['VehBrand', 'VehGas'])
)
show(effects.loc[chosen_points,
     ['model_id', 'feature', 'value', 'relativity', 'exposure_at_exact_value', 'anchor']])
display(Image(filename=str(ROOT / 'reports/s4/figures/relativitaeten.png')))
display(Image(filename=str(ROOT / 'reports/s4/figures/konzentration.png')))

## Aussagegrenzen und nächster Schritt

Ein niedrigerer Entwicklungs-OOF-Score belegt einen prädiktiven Unterschied innerhalb dieses Protokolls. Er beweist weder eine wahre Poisson-Verteilung, einen kausalen Risikotreiber noch einen marktfähigen Tarif. Konvergenz, Rang, Strafskalierung, effektive Freiheitsgrade und unbekannte Kategorien stehen im gespeicherten Modellnachweis; die Hyperparameter verschiedener Modellfamilien sind numerisch nicht gleichzusetzen.

Die unabhängige Schadenhöhen- und Risikoprämienbewertung folgt erst nach dieser gesperrten Entwicklungsentscheidung. Frequenz allein bewertet keine Euroverluste. Die jährliche reine Risikoprämie und erwartete Kosten der tatsächlichen Vertragsperiode müssen später getrennt ausgewiesen werden.